# Visualize Cohort Statistics

- Creator: Arpit Merchant (am84@sanger.ac.uk).
- Date of Creation: 20.01.2025
- Date of Last Modification: 20.01.2025 (Sebastian Birk; sebastian.birk@helmholtz-munich.de)

## 1. Setup

### 1.1 Import Libraries

In [ ]:
import os

import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import ConnectionPatch
import matplotlib.gridspec as gridspec
import warnings

warnings.filterwarnings("ignore")

### 1.2 Run Notebook Setup

### 1.3 Configure Paths and Create Directories

In [ ]:
# Define paths
st_data_folder_path = "/lustre/scratch126/cellgen/team361/DATASETS" # spatially resolved transcriptomics data
st_data_bronze_folder_path = f"{st_data_folder_path}/bronze"
st_data_silver_folder_path = f"{st_data_folder_path}/silver"
st_data_gold_folder_path = f"{st_data_folder_path}/gold"
st_data_url_folder_path = f"{st_data_folder_path}/urls"

artifacts_folder_path = '../../artifacts'
data_artifacts_folder_path = '../../artifacts/data'

# Create required directories
os.makedirs(st_data_bronze_folder_path, exist_ok=True)
os.makedirs(st_data_silver_folder_path, exist_ok=True)
os.makedirs(st_data_gold_folder_path, exist_ok=True)
os.makedirs(st_data_url_folder_path, exist_ok=True)

os.makedirs(data_artifacts_folder_path, exist_ok=True)

## 2. 60m Human Pre-training Pan-Organ Corpus 

In [ ]:
df = pd.read_csv(f'{data_artifacts_folder_path}/human_cohort1_60m_statistics.csv')
display(df.head())

num_datasets = df['dataset_id'].nunique()
print(f"Number of Datasets: {num_datasets}")

num_cells = df['n_cells'].sum()
print(f"Number of Cells: {num_cells}")

### Datasets with Tokenization Errors

### Statistics of GBM Data from Omer

In [ ]:
gbm_df = df[df['dataset_name'] == 'xhb1002-76b_3p']

In [ ]:
def get_donor_id(filename):
    filename = Path(filename).name
    return filename.split('-')[0].split('_')[0]

gbm_df['donor_id'] = gbm_df['adata_batch_path'].apply(get_donor_id)
gbm_donor_summary = gbm_df.groupby('donor_id')['n_cells'].sum().reset_index()
gbm_donor_summary = gbm_donor_summary.sort_values(by='n_cells', ascending=False)
display(gbm_donor_summary)

### Full Corpus Distribution

In [ ]:
ASSAY_NAME_MAP = {
    'xenium': 'Xenium',
    'merfish': 'MERFISH',
    'cosmx': 'CosMx',
    'cartana': 'ISS CARTANA',
    'starmap': 'STARmap'
}

def assay_distribution(
    num_cells,
    num_datasets,
    df,
    xname='percentage',
):
    # ----------------Process DataFrame----------------
    
    assay_counts = df.groupby("assay")['n_cells'].sum().reset_index()
    assay_counts['percentage'] = (assay_counts['n_cells'] / num_cells) * 100
    assay_counts = assay_counts.sort_values(by='n_cells', ascending=False)
    assay_counts['assay'] = assay_counts['assay'].apply(lambda x: ASSAY_NAME_MAP[x])
    
    # ----------------Plot Settings----------------
    plt.rcParams.update({
        'font.size': 20,
        'font.family': 'sans-serif',
    })
    _, ax = plt.subplots(1, 1, figsize=(6, 6))
    colors = sns.color_palette('husl', len(assay_counts))
    
    # ----------------Draw Figure----------------
    
    sns.barplot(
        data=assay_counts,
        x=xname,
        y="assay",
        hue='assay',
        legend=False,
        palette=colors,
        ax=ax,
        )
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # ----------------Annotations----------------

    n_cells_assay = list(assay_counts['n_cells'])
    for i, container in enumerate(ax.containers):
        ax.bar_label(container, labels=[f'{(n_cells_assay[i]/1e6):.2f}M'], label_type='edge', padding=5)

    # ----------------Labels----------------
    
    if xname == 'percentage':
        xlabel = 'Distribution (in %)'
    elif xname == 'n_cells':
        xlabel = 'Number of Cells'
    ax.set_xlabel(f"{xlabel}")
    ax.set_ylabel('')
    plt.suptitle(f"{num_cells/1e6:.2f}M Cells | {num_datasets} Datasets", fontsize=20)
    plt.savefig(f"{data_artifacts_folder_path}/human_cohort1_60m_assaydistribution.svg", dpi=300, bbox_inches='tight')

    # ----------------Visualize----------------
    plt.show()


assay_distribution(
    num_cells,
    num_datasets,
    df,
    xname='percentage',
    )

In [ ]:
tissue_counts['tissue']

In [ ]:
def tissue_distribution(
    num_cells,
    num_datasets,
    df,
    xname='percentage',
):
    # ----------------Process DataFrame----------------
    
    tissue_counts = df.groupby("tissue")['n_cells'].sum().reset_index()
    tissue_counts['percentage'] = (tissue_counts['n_cells'] / num_cells) * 100
    tissue_counts = tissue_counts.sort_values(by='n_cells', ascending=False)
    tissue_counts['tissue'] = tissue_counts['tissue'].str.capitalize()
    tissue_counts['tissue'] = tissue_counts['tissue'].replace('Spinal_cord', 'Spinal cord')
    tissue_counts['tissue'] = tissue_counts['tissue'].replace('Male_reproductive_system', 'Pelvis')

    # ----------------Plot Settings----------------
    plt.rcParams.update({
        'font.size': 20,
        'font.family': 'sans-serif',
    })
    _, ax = plt.subplots(1, 1, figsize=(6, 10))
    colors = sns.color_palette('husl', len(tissue_counts))
    
    # ----------------Draw Figure----------------
    
    sns.barplot(
        data=tissue_counts,
        x=xname,
        y="tissue",
        hue='tissue',
        legend=False,
        palette=colors,
        ax=ax,
        )
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # ----------------Annotations----------------

    n_cells_tissue = list(tissue_counts['n_cells'])
    for i, container in enumerate(ax.containers):
        ax.bar_label(container, labels=[f'{(n_cells_tissue[i]/1e6):.2f}M'], label_type='edge', padding=5)

    # ----------------Labels----------------
    
    if xname == 'percentage':
        xlabel = 'Distribution (in %)'
    elif xname == 'n_cells':
        xlabel = 'Number of Cells'
    ax.set_xlabel(f"{xlabel}")
    ax.set_ylabel('')
    plt.suptitle(f"{num_cells/1e6:.2f}M Cells | {num_datasets} Datasets", fontsize=14)
    plt.savefig(f"{data_artifacts_folder_path}/human_cohort1_60m_tissuedistribution.svg", dpi=300, bbox_inches='tight')

    # ----------------Visualize----------------
    plt.show()


tissue_distribution(
    num_cells,
    num_datasets,
    df,
    xname='percentage',
    )

In [ ]:
df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_density(df):
    # Create a figure with 2 subplots
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Density plot for n_cells
    sns.kdeplot(
        data=df,
        x="n_cells",
        fill=True,        # Fill under the curve
        common_norm=False, # If you don't want all densities to sum to 1
        ax=axes[0]
    )
    axes[0].set_title("Density of n_cells")
    axes[0].set_xlabel("Number of Cells")
    axes[0].set_ylabel("Density")

    # Density plot for n_genes
    sns.kdeplot(
        data=df,
        x="n_genes",
        fill=True,
        common_norm=False,
        ax=axes[1]
    )
    axes[1].set_title("Density of n_genes")
    axes[1].set_xlabel("Number of Genes")
    axes[1].set_ylabel("Density")

    # Adjust layout
    plt.tight_layout()
    plt.show()

# Usage:
plot_density(df)

## 30M Corpus Summary Statistics Visualization

In [ ]:
df = pd.read_csv("/lustre/scratch126/cellgen/team361/am84/NicheJEPA/results/silver_statistics/full_statistics.csv")
print(df.shape)
display(df.head())

In [ ]:
total_cells = df['n_cells'].sum()
print(f"Total number of cells: {total_cells}")

df_mouse = df[df['species'] == 'mus_musculus']
mouse_cells = df_mouse['n_cells'].sum()
print(f"Number of mouse cells: {mouse_cells}")
print(f"Number of human cells: {total_cells - mouse_cells}")


In [ ]:
df1 = df[df['dataset_folder'] == 'xhb1002-76b_1p']
df1

In [ ]:
df1['n_cells'] = df1['n_cells'].astype(int)
print(df1['n_cells'].sum())

In [ ]:
df1['n_genes'].value_counts()

In [ ]:
def plot_assay_sex_condition_species(df):
    fig, ax = plt.subplots(1,4, figsize=(24, 5))
    cols = ['assay', 'sex', 'disease_level1', 'species']
    colnames = ['Assay', 'Sex', 'Condition', 'Species']
    for i in range(4):
        col_counts = df.groupby(cols[i])['n_cells'].sum().reset_index()

        total_cells = col_counts['n_cells'].sum()
        col_counts['percentage'] = (col_counts['n_cells'] / total_cells) * 100

        sns.barplot(data=col_counts, x=cols[i], y='percentage', ax=ax[i])
        ax[i].set_title(f"{colnames[i]}")
        ax[i].set_xlabel('')
        ax[i].set_ylabel('Percentage of Total Cells')
        ax[i].tick_params(axis='x', rotation=15)

    plt.suptitle(f"Total Cells: {total_cells/1e6:.2f} million", fontsize=16)
    plt.show()

In [ ]:
def plot_tissue(df, species='homo_sapiens'):

    # Group by tissue and sum the number of cells
    df.columns = df.columns.str.capitalize()
    tissue_counts = df.groupby('Tissue')['N_cells'].sum().rename('Cells')
    tissue_counts = tissue_counts.sort_values(ascending=False)
    tissue_counts = tissue_counts.rename(index=lambda x: x.replace('_', ' ').capitalize())

    tissues = [str(s) for s in tissue_counts.index.tolist()]
    tissue_counts = tissue_counts.to_list()
    n_cells = sum(tissue_counts)
    ratios = [c/n_cells for c in tissue_counts]

    others_idx = next((i for i, ratio in enumerate(ratios) if ratio < 0.05), len(ratios)) - 1
    print(others_idx)
    
    tissue_categories = tissues[:-others_idx] + ['Others']
    tissue_category_counts = tissue_counts[:-others_idx] + [sum(tissue_counts[-others_idx:])]
    print(tissue_categories)
    print(tissue_category_counts)

    other_tissues = tissues[-others_idx:]
    other_ratios = ratios[-others_idx:]
    
    # make figure and assign axis objects
    font_size = 14
    plt.rcParams['font.size'] = font_size
    fig = plt.figure(figsize=(9, 9))
    gs = gridspec.GridSpec(2, 3, height_ratios=[1, 1], width_ratios=[1, 4, 1])
    ax1 = fig.add_subplot(gs[0, :])  # Pie chart occupies the top two columns
    ax2 = fig.add_subplot(gs[1, 1])  # Bar chart occupies the bottom-right column
    fig.subplots_adjust(wspace=0)
    fig.subplots_adjust(hspace=-0.3)  # Reduce vertical spacing

    # pie chart parameters
    angle = -67.5
    explode = [int(s=='Others')/10 + 0.02 for s in tissue_categories]
    # Set color palette for pie chart
    colors = sns.color_palette('Paired', len(tissue_categories))
    colors = colors[2:] + colors[:2]
    print(colors[0] == colors[-1])
    print(len(colors), len(tissue_categories), len(tissue_category_counts))
    ax1.set_prop_cycle('color', colors)
    wedges, *_ = ax1.pie(tissue_category_counts, autopct='%1.1f%%', startangle=angle,
                        labels=tissue_categories, explode=explode)
    # use ConnectionPatch to draw lines between the two plots
    theta1, theta2 = wedges[-1].theta1, wedges[-1].theta2
    center, r = wedges[-1].center, wedges[-1].r    

    # Horizontal bar chart parameters
    left = 0  # Starting point for horizontal bars
    bar_height = .2  # Thickness of each bar

    # Adding from the left matches the legend
    bar_width = 0.0
    below_2_bars = []  # To keep track of bars with ratios < 2%
    below_2_bars_width = 0  # Combined width for bars < 2%
    for j, (ratio, label) in enumerate(reversed([*zip(other_ratios, other_tissues)])):
        width = ratio / 10
        print(width, ratio)
        bar = ax2.barh(0, width, bar_height, left=left, color=colors[-1], label=label, alpha=0.1 + 0.1 * j)
        label_type = 'center'
        if ratio > 0.02:
            ax2.bar_label(bar, labels=[f"{ratio:.1%}"], label_type=label_type)
        else:
            # place just one text label for all labels below 2% in the middle their bars combined
            below_2_bars.append(bar)  # Add bars < 2% to a separate list
            below_2_bars_width += width
            
        left += width
        bar_width += width

    # Add a single label "< 2%" for all bars less than 2%
    if below_2_bars:
        ax2.text(below_2_bars_width / 2, bar_height / 2 - 0.1, "< 2%", ha='center', va='center', color='black', fontsize=font_size)        

    # Update ax2 aesthetics for horizontal layout
    ax2.legend()
    ax2.axis('off')
    ax2.set_ylim(-2.5 * bar_height, 2.5 * bar_height)

    # Add the legend below the bar plot
    ax2.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 0.4),
        ncol=2,
        frameon=False
    )

    # Top-left connection line (to left edge of the bar plot)
    x = r * np.cos(np.pi / 180 * theta1) + center[0]
    y = r * np.sin(np.pi / 180 * theta1) + center[1]
    con = ConnectionPatch(xyA=(0, 0.1), coordsA=ax2.transData,
                        xyB=(x, y), coordsB=ax1.transData)
    con.set_color([0, 0, 0])
    con.set_linewidth(1)
    ax2.add_artist(con)

    # Top-right connection line (to right edge of the bar plot)
    x = r * np.cos(np.pi / 180 * theta2) + center[0]
    y = r * np.sin(np.pi / 180 * theta2) + center[1]
    con = ConnectionPatch(xyA=(bar_width, 0.1), coordsA=ax2.transData,
                        xyB=(x, y), coordsB=ax1.transData)
    con.set_color([0, 0, 0])
    con.set_linewidth(1)
    ax2.add_artist(con)

    # plt.suptitle(f"Percentage of Cells by Tissue | Species: {species} | Total Cells: {n_cells/1e6:.1f} million")
    plt.savefig(f"tissue_distribution_{species}.png", bbox_inches='tight', dpi=300)

In [ ]:
# Plot select df for species as homo_sapiens
species = 'homo_sapiens'
df_human = df[df['species'] == species]
# plot_assay_sex_condition_species(df_human)
plot_tissue(df_human, species=species)

In [ ]:
# tissue_counts = tissue_counts.rename(index=lambda x: x.replace('_', ' ').capitalize())

# display(df_human)

print(tissue_counts)
counts = tissue_counts.to_list()
print(counts)

In [ ]:
# Plot select df for species as mus_musculus
df_mouse = df[df['species'] == 'mus_musculus']
plot_assay_sex_condition_species(df_mouse)
plot_tissue(df_mouse)

In [ ]:
# Group by assay and tissue and sum the number of cells
assay_tissue_counts = df.groupby(['assay', 'tissue'])['n_cells'].sum().reset_index()

# Calculate the total number of cells for each assay
total_cells_per_assay = assay_tissue_counts.groupby('assay')['n_cells'].sum().reset_index()
total_cells_per_assay.columns = ['assay', 'total_cells']

# Merge the total cells per assay back to the assay_tissue_counts
assay_tissue_counts = pd.merge(assay_tissue_counts, total_cells_per_assay, on='assay')

# Calculate the proportion of each tissue within each assay
assay_tissue_counts['proportion'] = (assay_tissue_counts['n_cells'] / assay_tissue_counts['total_cells']) * 100

# Plot bar chart
plt.figure(figsize=(12, 8))
barplot = sns.barplot(data=assay_tissue_counts, x='assay', y='proportion', hue='tissue')
plt.title('Proportion of Tissue within Each Assay')
plt.xlabel('Assay')
plt.ylabel('Proportion of Total Cells (%)')
plt.legend(title='Tissue')

idx_largest = {}
for i in range(len(barplot.patches)):
    bar = barplot.patches[i]
    bar_height = bar.get_height()
    if len(idx_largest) < 11:
        idx_largest[i] = bar_height
    else:
        # Remove item from idx_largest with smallest value
        smallest_value = min(idx_largest.values())
        if bar_height > smallest_value:
            smallest_key = min(idx_largest, key=idx_largest.get)
            del idx_largest[smallest_key]
            idx_largest[i] = bar_height

sorted_idx_largest = sorted(idx_largest, key=idx_largest.get, reverse=True)
names = ['cns', 'multicellular_organism', 'brain', 'liver', 'lung', 'brain', 'bone','uterus', 'brain', 'prostate', 'skin']
assert len(names) == len(sorted_idx_largest)

for i, idx in enumerate(sorted_idx_largest):
    bar = barplot.patches[idx]
    print(names[i], bar.get_height())
    barplot.annotate(
        names[i],
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha='center',
        va='bottom',
        xytext=(0, 10),  # Offset text position
        textcoords='offset points',
        arrowprops=dict(arrowstyle="->", color='black')
    )

plt.show()